# 3. Recover an SIS Einstein radius

This notebook performs the current end-to-end inference problem. We simulate a noisy PSF-convolved observation, hold the Gaussian source fixed, and infer the SIS Einstein radius.

The production renderer and likelihood are differentiable. Project Akira passes their JAX gradient to SciPy's bounded L-BFGS-B optimizer.

In [ ]:
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

from akira import (
    CartesianGrid, Gaussian, SIS, fit_sis_einstein_radius, gaussian_log_likelihood,
    gaussian_psf, render_sis_gaussian, simulate_imaging,
)
from akira.imaging import convolve_psf

In [ ]:
grid = CartesianGrid(shape=(100, 100), pixel_scale=0.04, supersampling=2)
true_lens = SIS(einstein_radius=1.0)
source = Gaussian(
    intensity=1.0, sigma=0.09, centre=(0.08, -0.04), q=0.75, phi=0.3
)
psf = gaussian_psf(sigma_pixels=1.0, size=9)
observation = simulate_imaging(
    render_sis_gaussian(grid, true_lens, source),
    psf=psf,
    noise_sigma=0.02,
    pixel_scale=grid.pixel_scale,
    seed=42,
)

## Inspect the likelihood surface

For teaching purposes we scan one parameter explicitly. The optimizer does not require this scan; it uses the likelihood and its gradient directly.

In [ ]:
def log_likelihood_for_radius(einstein_radius):
    model = render_sis_gaussian(grid, SIS(float(einstein_radius)), source)
    model = convolve_psf(model, observation.psf.kernel)
    return gaussian_log_likelihood(
        observation.data, model, observation.noise_map, observation.mask
    )

trial_radii = np.linspace(0.85, 1.15, 61)
trial_log_likelihoods = np.array([float(log_likelihood_for_radius(r)) for r in trial_radii])

plt.figure(figsize=(7, 4))
plt.plot(trial_radii, trial_log_likelihoods - trial_log_likelihoods.max())
plt.axvline(true_lens.einstein_radius, color='black', linestyle='--', label='Truth')
plt.xlabel('Einstein radius [arcsec]')
plt.ylabel(r'$\Delta \log \mathcal{L}$')
plt.legend()
plt.show()

## Examine a JAX gradient

The class constructors perform Python-side validation, so the differentiable kernel is called directly when taking derivatives. This is the same numerical path used by the fitter.

In [ ]:
from akira.render import render_sis_gaussian_from_coordinates

coordinates = grid.subpixel_coordinates()
def differentiable_log_likelihood(radius):
    unconvolved = render_sis_gaussian_from_coordinates(
        coordinates, radius, jnp.array([0.0, 0.0]),
        source.intensity, source.sigma, jnp.asarray(source.centre), source.q, source.phi,
    )
    model = convolve_psf(unconvolved, observation.psf.kernel)
    return gaussian_log_likelihood(
        observation.data, model, observation.noise_map, observation.mask
    )

initial_radius = 0.8
initial_gradient = jax.grad(differentiable_log_likelihood)(initial_radius)
print(f'Likelihood gradient at {initial_radius:.2f} arcsec: {float(initial_gradient):.3f}')
print('A positive gradient means increasing the radius improves the likelihood here.')

## Fit the Einstein radius

In [ ]:
result = fit_sis_einstein_radius(
    observation=observation,
    grid=grid,
    source=source,
    initial_einstein_radius=initial_radius,
    bounds=(0.5, 1.5),
)

print(f'True Einstein radius:      {true_lens.einstein_radius:.6f} arcsec')
print(f'Recovered Einstein radius: {result.einstein_radius:.6f} arcsec')
print(f'Absolute error:            {abs(result.einstein_radius - true_lens.einstein_radius):.6f} arcsec')
print(f'Optimizer success:         {result.success}')
print(f'Likelihood evaluations:    {result.evaluations}')
assert result.success
assert abs(result.einstein_radius - true_lens.einstein_radius) < 0.002

In [ ]:
residual = np.asarray(observation.data - result.model_image)
normalized_residual = residual / np.asarray(observation.noise_map)

fig, axes = plt.subplots(1, 3, figsize=(13, 4), constrained_layout=True)
panels = [
    (observation.data, 'Simulated data', 'magma'),
    (result.model_image, 'Best-fit model', 'magma'),
    (normalized_residual, 'Normalized residual', 'coolwarm'),
]
for ax, (image, title, cmap) in zip(axes, panels):
    display = ax.imshow(np.asarray(image), origin='lower', cmap=cmap)
    ax.set_title(title)
    fig.colorbar(display, ax=ax, shrink=0.82)
plt.show()